# SET UP

In [ ]:
# ALL IMPORTS

import json
import os
from collections import defaultdict

from torch.utils.data import Dataset
from PIL import Image
import torchvision.transforms as transforms
import torch

import matplotlib.pyplot as plt
import numpy as np

import cv2
from torch.utils.data import DataLoader


In [ ]:
# mount our Google Drive

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# unzip images in Google Drive

!unzip '/content/drive/MyDrive/Colab Notebooks/APS360/Project/chessred2k.zip' -d '/content/drive/MyDrive/Colab Notebooks/APS360/Project/ChessReD2K'

In [ ]:
# unzip images in Google Drive

!unzip '/content/drive/MyDrive/Colab Notebooks/APS360/Project/chessred.zip' -d '/content/drive/MyDrive/Colab Notebooks/APS360/Project/ChessReD'

In [ ]:
data_dir = '/content/drive/MyDrive/Colab Notebooks/APS360/Project/ChessReD'
annotation_dir = '/content/drive/MyDrive/Colab Notebooks/APS360/Project/annotations.json'

with open(annotation_dir, "r") as f:
            annotations = json.load(f)

In [ ]:
# messing around with annotations.json

print(annotations.keys(), "\n")
print(annotations['categories'], "\n") # id map for each type of piece
print(annotations['splits']['train']['image_ids'])
print(annotations['splits']['chessred2k']['train']['image_ids'])
# print(annotations['splits']['chessred2k']['val'])
# print(annotations['splits']['chessred2k']['test'])

# print((annotations['splits']['chessred2k']['train']['image_ids']))

# print(annotations['annotations']['pieces'][0]) # all 223,804 piece annotations for 10800 images
# print(annotations['annotations']['corners'][0])

# print(annotations['images'][0])

# CONVERTING ANNOTATIONS TO FEN

In [ ]:
# IMPORTANT

image_id_to_path = {img['id']: os.path.join(data_dir, img['path']) for img in annotations['images']}
print(image_id_to_path)

image_to_pieces = defaultdict(list)
for ann in annotations['annotations']['pieces']:
    image_to_pieces[ann['image_id']].append(ann)
print(image_to_pieces[0])

print(image_id_to_path[103])

print(image_to_pieces[8000])

In [ ]:
# IMPORTANT
# uppercase is white, lowercase is black
id_to_piece = {
    0: 'P', 1: 'R', 2: 'N', 3: 'B', 4: 'Q', 5: 'K',
    6: 'p', 7: 'r', 8: 'n', 9: 'b', 10: 'q', 11: 'k',
    12: '1'  # empty
}

piece_to_label = {
    'P': 0, 'R': 1, 'N': 2, 'B': 3, 'Q': 4, 'K': 5,
    'p': 6, 'r': 7, 'n': 8, 'b': 9, 'q': 10, 'k': 11
}

def fen_to_label_vector(fen, empty_char='0'):
    squares = []
    for row in fen.split('/'):
        for ch in row:
            if ch == empty_char:
                squares.append(12)
            elif ch.isdigit():
                squares.extend([12] * int(ch))  # for compressed digits like 8
            else:
                squares.append(piece_to_label[ch])
    assert len(squares) == 64, f"Expected 64 squares but got {len(squares)} in FEN: {fen}"
    return torch.tensor(squares, dtype=torch.long)

# convert image piece list to FEN
def pieces_to_fen(piece_list):
    board = [['1'] * 8 for _ in range(8)]
    pos_to_index = lambda pos: (8 - int(pos[1]), ord(pos[0]) - ord('a'))

    for piece in piece_list:
        row, col = pos_to_index(piece['chessboard_position'])
        board[row][col] = id_to_piece[piece['category_id']]

    fen_rows = []
    for row in board:
        fen_row = ''
        count = 0
        for cell in row:
            if cell == '1':
                fen_row += '0'
            else:
                fen_row += cell
        fen_rows.append(fen_row)

    return '/'.join(fen_rows)
def label_vector_to_fen(label_vector):
    """
    Converts a length-64 label vector into a FEN string.
    Assumes label 12 is empty.
    """
    assert len(label_vector) == 64, f"Expected 64 squares, got {len(label_vector)}"

    fen_rows = []
    for i in range(0, 64, 8):
        row = label_vector[i:i+8]
        fen_row = ''
        empty_count = 0

        for val in row:
            if val == 12:  # empty square
                empty_count += 1
            else:
                if empty_count > 0:
                    fen_row += str(empty_count)
                    empty_count = 0
                fen_row += id_to_piece[int(val)]

        if empty_count > 0:
            fen_row += str(empty_count)

        fen_rows.append(fen_row)

    return '/'.join(fen_rows)

print(pieces_to_fen(image_to_pieces[2384]))
print(fen_to_label_vector(pieces_to_fen(image_to_pieces[10])))
print(fen_to_label_vector(pieces_to_fen(image_to_pieces[10])).shape)

# VISUALIZE DATASET

In [ ]:
# NOT IMPORTANT

data_transform = transforms.Compose([
      transforms.Resize((224,224)),
      transforms.ToTensor(),
      ])


# dataset class to be used by DataLoader. Needs init, len, getitem functions
class ChessFENDataset(Dataset):
    def __init__(self, image_ids, image_to_pieces, image_id_to_path, transform=None):
        self.image_ids = image_ids
        self.image_to_pieces = image_to_pieces
        self.image_id_to_path = image_id_to_path
        self.transform = transform or transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor()
        ])

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        image_id = self.image_ids[idx]
        image_path = self.image_id_to_path[image_id]
        image = Image.open(image_path).convert("RGB")
        image = self.transform(image)

        piece_list = self.image_to_pieces[image_id]
        fen = pieces_to_fen(piece_list)
        label = fen_to_label_vector(fen)
        return image, label


In [ ]:
# NOT IMPORTANT, JUST FOR GETTING SOME NUMBERS

train_ids = annotations['splits']['chessred2k']['train']['image_ids']
train_dataset = ChessFENDataset(train_ids, image_to_pieces, image_id_to_path)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=32, shuffle=True)

print(f"ChessReD2K Training Images: {len(train_dataset)}")

val_ids = annotations['splits']['chessred2k']['val']['image_ids']
val_dataset = ChessFENDataset(val_ids, image_to_pieces, image_id_to_path)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=32, shuffle=True)

print(f"ChessReD2K Validation Images: {len(val_dataset)}")

test_ids = annotations['splits']['chessred2k']['test']['image_ids']
test_dataset = ChessFENDataset(test_ids, image_to_pieces, image_id_to_path)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=32, shuffle=True)

print(f"ChessReD2K Testing Images: {len(test_dataset)}")


train_ids = annotations['splits']['train']['image_ids']
train_dataset = ChessFENDataset(train_ids, image_to_pieces, image_id_to_path)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=32, shuffle=True)

print(f"ChessReD Training Images: {len(train_dataset)}")

val_ids = annotations['splits']['val']['image_ids']
val_dataset = ChessFENDataset(val_ids, image_to_pieces, image_id_to_path)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=32, shuffle=True)

print(f"ChessReD Validation Images: {len(val_dataset)}")

test_ids = annotations['splits']['test']['image_ids']
test_dataset = ChessFENDataset(test_ids, image_to_pieces, image_id_to_path)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=32, shuffle=True)

print(f"ChessReD Testing Images: {len(test_dataset)}")



In [ ]:
# NOT IMPORTANT

# obtain one batch of training images
dataiter = iter(train_loader)
images, labels = next(dataiter)
images = images.numpy() # convert images to numpy for display

# plot the images in the batch, along with the corresponding labels
fig = plt.figure(figsize=(25, 4))
for idx in np.arange(1):
    ax = fig.add_subplot(2, int(20/2), idx+1, xticks=[], yticks=[])
    img = images[idx]
    plt.imshow(np.transpose(img, (1, 2, 0)))
    plt.title(labels[idx])

In [ ]:
#NOT IMPORTANT

import matplotlib.image as mpimg
img = mpimg.imread(image_id_to_path[0])
fig, ax = plt.subplots()
ax.imshow(img)
plt.show()


# PREPROCESSING (HOUGH), SAVING TO DRIVE

In [ ]:
# VERY IMPORTANT, PREPROCESSING OF BOARD TO GET OVERHEAD IMG

import cv2
import numpy as np
import matplotlib.pyplot as plt

def detect_board_corners(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    edges = cv2.Canny(blur, 50, 150)

    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5))
    dilated = cv2.dilate(edges, kernel, iterations=2)

    contours, _ = cv2.findContours(dilated, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    contours = sorted(contours, key=cv2.contourArea, reverse=True)

    image_area = img.shape[0] * img.shape[1]

    for cnt in contours:
        epsilon = 0.02 * cv2.arcLength(cnt, True)
        approx = cv2.approxPolyDP(cnt, epsilon, True)

        if len(approx) == 4 and cv2.contourArea(approx) > 0.1 * image_area:
            corners = np.float32([pt[0] for pt in approx])
            ordered = order_points_robust(corners)
            return ordered, dilated

    return None, dilated


def order_points_robust(pts):
    # First, get bounding box
    x_sorted = pts[np.argsort(pts[:, 0]), :]
    left_most = x_sorted[:2, :]
    right_most = x_sorted[2:, :]

    # sort by y-coord
    tl, bl = left_most[np.argsort(left_most[:, 1]), :]
    tr, br = right_most[np.argsort(right_most[:, 1]), :]

    return np.array([tl, tr, br, bl], dtype="float32")  # Clockwise

def draw_corners(img, corners):
    img_copy = img.copy()
    labels = ["TL", "TR", "BR", "BL"]
    for i, pt in enumerate(corners):
        pt_int = tuple(np.int32(pt))
        cv2.circle(img_copy, pt_int, 14, (0, 0, 255), -1)
        cv2.putText(img_copy, labels[i], (pt_int[0] + 10, pt_int[1] - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 2.0, (255, 0, 0), 3)
    for i in range(4):
        pt1 = tuple(np.int32(corners[i]))
        pt2 = tuple(np.int32(corners[(i + 1) % 4]))
        cv2.line(img_copy, pt1, pt2, (0, 255, 0), 2)
    return img_copy


def warp_board(img, corners, output_size=400):
    src_pts = order_points_robust(corners)
    dst_pts = np.array([
        [0, 0],
        [output_size-1, 0],
        [output_size-1, output_size-1],
        [0, output_size-1]
    ], dtype="float32")

    M = cv2.getPerspectiveTransform(src_pts, dst_pts)
    warped = cv2.warpPerspective(img, M, (output_size, output_size))
    return warped

def slice_squares(warped, square_size=50):
    squares = []
    for row in range(8):
        for col in range(8):
            x1 = col * square_size
            y1 = row * square_size
            square = warped[y1:y1 + square_size, x1:x1 + square_size]
            squares.append(square)
    return squares

def preprocess_chessboard(image_path, output_size=400, display=True):
    img = cv2.imread(image_path)
    corners, debug_dilated = detect_board_corners(img)

    if corners is None:
        print("no corners")
        return
        raise ValueError("Board corners could not be detected.")

    corner_overlay = draw_corners(img, corners)
    warped = warp_board(img, corners, output_size)
    squares = slice_squares(warped, square_size=output_size // 8)

    if display:
        titles = ["Original", "Dilated Edges", "Corner Overlay", "Warped Top-Down"]
        images = [img, debug_dilated, corner_overlay, warped]

        plt.figure(figsize=(60, 20))
        for i in range(4):
            plt.subplot(1, 4, i + 1)
            img_disp = images[i]
            if len(img_disp.shape) == 2:
                plt.imshow(img_disp, cmap='gray')
            else:
                img_disp = cv2.cvtColor(img_disp, cv2.COLOR_BGR2RGB)
                plt.imshow(img_disp)
            plt.title(titles[i], fontsize=70)
            plt.axis('off')
        plt.tight_layout()
        plt.show()

    return warped, squares

test = preprocess_chessboard(image_id_to_path.get(annotations['splits']['chessred2k']['train']['image_ids'][10]))
print(pieces_to_fen(image_to_pieces[annotations['splits']['chessred2k']['train']['image_ids'][2]]))

In [ ]:
def expand_fen(fen, empty_char='0'):
    board = []
    for row in fen.split('/'):
        expanded_row = []
        for ch in row:
            if ch.isdigit() and ch != empty_char:
                expanded_row.extend([empty_char] * int(ch))
            else:
                expanded_row.append(ch)
        board.append(expanded_row)
    return board

def compress_fen_row(row, empty_char='0'):
    compressed = ''
    count = 0
    for ch in row:
        if ch == empty_char:
            count += 1
        else:
            if count:
                compressed += str(count)
                count = 0
            compressed += ch
    if count:
        compressed += str(count)
    return compressed

def board_to_fen(board, empty_char='0'):
    return '/'.join([compress_fen_row(row, empty_char=empty_char) for row in board])

def rotate_fen_90(fen, empty_char='0'):
    board = expand_fen(fen, empty_char)
    rotated = list(zip(*board[::-1]))
    rotated = [list(row) for row in rotated]
    return board_to_fen(rotated, empty_char)

def rotate_fen_180(fen, empty_char='0'):
    board = expand_fen(fen, empty_char)
    rotated = [row[::-1] for row in board[::-1]]
    return board_to_fen(rotated, empty_char)

def rotate_fen_270(fen, empty_char='0'):
    board = expand_fen(fen, empty_char)
    rotated = list(zip(*board))[::-1]
    rotated = [list(row) for row in rotated]
    return board_to_fen(rotated, empty_char)
test_fen = "rnbqkbnr/pppppppp/00000000/00000000/00000000/00000000/PPPPPPPP/RNBQKBNR"

print("Original:     ", test_fen)
print("Rotated 90°:  ", rotate_fen_90(test_fen))
print("Rotated 180°: ", rotate_fen_180(test_fen))
print("Rotated 270°: ", rotate_fen_270(test_fen))


## PREPROCESS CHESSRED TRAIN

In [ ]:
# VERY IMPORTANT, SAVE WARPED CHESSRES2K IMAGES to DRIVE

import os
from google.colab import drive
from tqdm.notebook import tqdm


# Create save directory if it doesn't exist
save_dir = "/content/drive/MyDrive/Colab Notebooks/APS360/Project/ChessReD_Hough"
os.makedirs(save_dir, exist_ok=True)

X = []  # Paths to warped images
y = []  # Labels (4 FEN rotations each)

valid_ids = []
error_ids = []
invalid_warp_ids = []

# train_chessred2k_ids = annotations['splits']['chessred2k']['train']['image_ids']
chessred_ids = annotations['splits']['train']['image_ids']

for i in tqdm(chessred_ids, desc="Processing Images"):
    path = image_id_to_path.get(i)
    if path is None:
        error_ids.append(i)
        continue

    try:
        warped, _ = preprocess_chessboard(path, display=False)
        if warped is None or np.std(warped) < 50:
            invalid_warp_ids.append(i)
            continue

        # Save warped image
        img_save_path = os.path.join(save_dir, f"{i}.png")
        cv2.imwrite(img_save_path, warped)

        # Get FEN
        pieces = image_to_pieces[i]
        fen = pieces_to_fen(pieces)

        X.append(img_save_path)
        y.append(fen)
        valid_ids.append(i)

    except Exception as e:
        error_ids.append(i)


with open("/content/drive/MyDrive/Colab Notebooks/APS360/Project/chessred_hough.pkl", "wb") as f:
  pickle.dump((X, y), f)

print("✅ Total valid images:", len(valid_ids))
print("❌ Total errors:", len(error_ids))
print("⚠️ Total invalid warped:", len(invalid_warp_ids))


## PREPROCESS CHESSRED VAL

In [ ]:
X = []  # Paths to warped images
y = []  # Labels (4 FEN rotations each)

valid_ids = []
error_ids = []
invalid_warp_ids = []

# train_chessred2k_ids = annotations['splits']['chessred2k']['train']['image_ids']
chessred_ids = annotations['splits']['val']['image_ids']

for i in tqdm(chessred_ids, desc="Processing Images"):
    path = image_id_to_path.get(i)
    if path is None:
        error_ids.append(i)
        continue

    try:
        warped, _ = preprocess_chessboard(path, display=False)
        if warped is None or np.std(warped) < 50:
            invalid_warp_ids.append(i)
            continue

        # Save warped image
        img_save_path = os.path.join(save_dir, f"{i}.png")
        cv2.imwrite(img_save_path, warped)

        # Get FEN
        pieces = image_to_pieces[i]
        fen = pieces_to_fen(pieces)

        X.append(img_save_path)
        y.append(fen)
        valid_ids.append(i)

    except Exception as e:
        error_ids.append(i)


with open("/content/drive/MyDrive/Colab Notebooks/APS360/Project/chessred_hough_val.pkl", "wb") as f:
  pickle.dump((X, y), f)

print("✅ Total valid images:", len(valid_ids))
print("❌ Total errors:", len(error_ids))
print("⚠️ Total invalid warped:", len(invalid_warp_ids))

## PREPROCESS CHESSRED TEST

In [ ]:
X = []  # Paths to warped images
y = []  # Labels (4 FEN rotations each)

valid_ids = []
error_ids = []
invalid_warp_ids = []

# train_chessred2k_ids = annotations['splits']['chessred2k']['train']['image_ids']
chessred_ids = annotations['splits']['test']['image_ids']

for i in tqdm(chessred_ids, desc="Processing Images"):
    path = image_id_to_path.get(i)
    if path is None:
        error_ids.append(i)
        continue

    try:
        warped, _ = preprocess_chessboard(path, display=False)
        if warped is None or np.std(warped) < 50:
            invalid_warp_ids.append(i)
            continue

        # Save warped image
        img_save_path = os.path.join(save_dir, f"{i}.png")
        cv2.imwrite(img_save_path, warped)

        # Get FEN
        pieces = image_to_pieces[i]
        fen = pieces_to_fen(pieces)

        X.append(img_save_path)
        y.append(fen)
        valid_ids.append(i)

    except Exception as e:
        error_ids.append(i)


with open("/content/drive/MyDrive/Colab Notebooks/APS360/Project/chessred_hough_test.pkl", "wb") as f:
  pickle.dump((X, y), f)

print("✅ Total valid images:", len(valid_ids))
print("❌ Total errors:", len(error_ids))
print("⚠️ Total invalid warped:", len(invalid_warp_ids))

In [ ]:
# # USED TO CONVERT IMAGES, LABELS TO DATASET

# from torch.utils.data import Dataset, DataLoader
# from torchvision import transforms
# from PIL import Image

# class ChessboardDataset(Dataset):
#     def __init__(self, image_paths, fen_labels, transform=None):
#         self.image_paths = image_paths
#         self.fen_labels = fen_labels
#         self.transform = transform or transforms.ToTensor()

#     def __len__(self):
#         return len(self.image_paths)

#     def __getitem__(self, idx):
#         img_path = self.image_paths[idx]
#         fen = self.fen_labels[idx]

#         # Load and transform image
#         image = Image.open(img_path).convert('RGB')
#         image_tensor = self.transform(image)

#         # Convert FEN string to label tensor (shape: [64])
#         label_tensor = fen_to_label_vector(fen)

#         return image_tensor, label_tensor

# DATASET

In [ ]:
class ChessboardRotDataset(Dataset):
    def __init__(self, image_paths, fen_labels, transform=None):
        self.image_paths = image_paths
        self.fen_labels = fen_labels
        self.transform = transform or transforms.ToTensor()

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
      img_path = self.image_paths[idx]
      fen = self.fen_labels[idx]

      image = Image.open(img_path).convert('RGB')
      image_tensor = self.transform(image)

      try:
          rotations = [
              fen_to_label_vector(fen),
              fen_to_label_vector(rotate_fen_90(fen)),
              fen_to_label_vector(rotate_fen_180(fen)),
              fen_to_label_vector(rotate_fen_270(fen)),
          ]
      except AssertionError as e:
          print(f"Bad FEN at index {idx}:\n{fen}")
          raise e

      return image_tensor, rotations

## TRAIN LOADER

In [ ]:
import pickle

with open("/content/drive/MyDrive/Colab Notebooks/APS360/Project/chessred_hough.pkl", 'rb') as f:
    X_loaded, y_loaded = pickle.load(f)
# print(X_loaded)
# print(y_loaded)
data_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomRotation(5),
    transforms.RandomResizedCrop(256, scale=(0.9, 1.0), ratio=(0.95, 1.05)),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor()
])
def custom_collate(batch):
    images, label_sets = zip(*batch)
    images = torch.stack(images, dim=0)
    return images, label_sets  # label_sets is a list of B lists of 4 tensors

# Recreate Dataset and DataLoader
train_dataset = ChessboardRotDataset(X_loaded, y_loaded, transform=data_transform)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, collate_fn=custom_collate)

print(f"Total images: {len(train_dataset)}")

## VAL LOADER

In [ ]:
import pickle

with open("/content/drive/MyDrive/Colab Notebooks/APS360/Project/chessred_hough_val.pkl", 'rb') as f:
    X_loaded, y_loaded = pickle.load(f)
print(X_loaded)
print(y_loaded)
data_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
])

# Recreate Dataset and DataLoader
val_dataset = ChessboardRotDataset(X_loaded, y_loaded, transform=data_transform)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=True, collate_fn=custom_collate)
print(f"Total images: {len(val_dataset)}")

## TEST LOADER (uses diff class)

In [ ]:
import pickle
class ChessboardRotDatasetTEST(Dataset):
    def __init__(self, image_paths, fen_labels, transform=None):
        self.image_paths = image_paths
        self.fen_labels = fen_labels
        self.transform = transform or transforms.ToTensor()

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
      img_path = self.image_paths[idx]
      fen = self.fen_labels[idx]

      image = Image.open(img_path).convert('RGB')
      image_tensor = self.transform(image)

      try:
          rotations = [
              fen_to_label_vector(fen),
              fen_to_label_vector(rotate_fen_90(fen)),
              fen_to_label_vector(rotate_fen_180(fen)),
              fen_to_label_vector(rotate_fen_270(fen)),
          ]
      except AssertionError as e:
          print(f"Bad FEN at index {idx}:\n{fen}")
          raise e
      image_path = self.image_paths[idx]

      return image_tensor, rotations, image_path
with open("/content/drive/MyDrive/Colab Notebooks/APS360/Project/chessred_hough_test.pkl", 'rb') as f:
    X_loaded, y_loaded = pickle.load(f)
print(X_loaded)
print(y_loaded)
data_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
])
def custom_collateTEST(batch):
    images, label_sets, paths = zip(*batch)  # Now unpack 3 values
    images = torch.stack(images, dim=0)
    return images, label_sets, paths
# def custom_collate(batch):
#     # Unpack only the first two values: image_tensor and rotations
#     images, label_sets, _ = zip(*batch)  # Ignore paths
#     images = torch.stack(images, dim=0)
#     return images, label_sets


# Recreate Dataset and DataLoader
test_dataset = ChessboardRotDatasetTEST(X_loaded, y_loaded, transform=data_transform)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=True, collate_fn=custom_collateTEST)
print(f"Total images: {len(test_dataset)}")

test_dataset_plain = ChessboardRotDataset(X_loaded, y_loaded, transform=data_transform)
test_loader_plain = DataLoader(test_dataset_plain, batch_size=16, shuffle=True, collate_fn=custom_collate)
print(f"Total images: {len(test_dataset_plain)}")


# TRAIN FUNCTION

In [ ]:
def train(
    model, dataloader, device, epochs=15,
    save_model=True, save_dir=None,
    plot_loss=True, lr=1e-4, weight_decay=1e-5,
    empty_class_idx=12, empty_class_weight=0.3,
    val_loader=None  # <- added
):
    import matplotlib.pyplot as plt

    model = model.to(device)

    weights = torch.ones(13)
    weights[empty_class_idx] = empty_class_weight
    weights = weights.to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    epoch_train_losses = []
    epoch_train_acc_all = []
    epoch_train_acc_no_empty = []

    epoch_val_losses = []
    epoch_val_acc_all = []
    epoch_val_acc_no_empty = []

    if save_model:
        assert save_dir is not None, "Please provide a save_dir to save model checkpoints."
        os.makedirs(save_dir, exist_ok=True)

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        correct_all = 0
        correct_non_empty = 0
        total_all = 0
        total_non_empty = 0

        for images, label_rotations in dataloader:
            images = images.to(device)
            label_rotations = [[lbl.to(device) for lbl in rotations] for rotations in label_rotations]

            optimizer.zero_grad()
            outputs = model(images)  # (B, 64, 13)

            losses = []
            matched_labels = []
            for i in range(images.size(0)):
                sample_losses = [F.cross_entropy(outputs[i], lbl, weight=weights, ignore_index=-1) for lbl in label_rotations[i]]
                best_idx = torch.argmin(torch.stack(sample_losses))
                losses.append(sample_losses[best_idx])
                matched_labels.append(label_rotations[i][best_idx])
            loss = torch.stack(losses).mean()
            matched_labels = torch.stack(matched_labels)

            loss.backward()
            optimizer.step()
            running_loss += loss.item()

            preds = outputs.argmax(dim=2)
            mask = matched_labels != empty_class_idx
            correct_all += (preds == matched_labels).sum().item()
            total_all += matched_labels.numel()
            correct_non_empty += ((preds == matched_labels) & mask).sum().item()
            total_non_empty += mask.sum().item()

        avg_train_loss = running_loss / len(dataloader)
        acc_all = correct_all / total_all if total_all > 0 else 0.0
        acc_no_empty = correct_non_empty / total_non_empty if total_non_empty > 0 else 0.0

        epoch_train_losses.append(avg_train_loss)
        epoch_train_acc_all.append(acc_all)
        epoch_train_acc_no_empty.append(acc_no_empty)

        print(f"Epoch [{epoch+1}/{epochs}] - Loss: {avg_train_loss:.4f} | Acc (all): {acc_all:.4f} | Acc (non-empty): {acc_no_empty:.4f}")

        if save_model:
            save_path = os.path.join(save_dir, f"epoch{epoch+1}.pth")
            torch.save(model.state_dict(), save_path)
            print(f"✅ Model saved to: {save_path}")

        # --- Evaluate on validation set (if provided) ---
        if val_loader is not None:
            val_loss, val_acc_all, val_acc_no_empty, _ = evaluate_rotation_invariant(model, val_loader, device=device)
            epoch_val_losses.append(val_loss)
            epoch_val_acc_all.append(val_acc_all)
            epoch_val_acc_no_empty.append(val_acc_no_empty)

    # --- Plotting ---
    if plot_loss:
        plt.figure(figsize=(12, 5))

        # Loss plot
        plt.subplot(1, 2, 1)
        plt.plot(epoch_train_losses, label='Train Loss')
        if val_loader is not None:
            plt.plot(epoch_val_losses, label='Val Loss')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.title('Loss Curve')
        plt.legend()

        # Accuracy plot
        plt.subplot(1, 2, 2)
        plt.plot(epoch_train_acc_all, label='Train Acc (All)')
        plt.plot(epoch_train_acc_no_empty, label='Train Acc (Non-Empty)')
        if val_loader is not None:
            plt.plot(epoch_val_acc_all, label='Val Acc (All)')
            plt.plot(epoch_val_acc_no_empty, label='Val Acc (Non-Empty)')
        plt.xlabel('Epoch')
        plt.ylabel('Accuracy')
        plt.title('Accuracy Curve')
        plt.legend()

        plt.tight_layout()
        plt.show()

# EVAL FUNCTION (SIMPLE)

In [ ]:
def evaluate_rotation_invariant(model, dataloader, device, empty_class_idx=12):
    import collections

    model.eval()
    total_loss = 0.0
    correct_all = 0
    correct_non_empty = 0
    total_all = 0
    total_non_empty = 0
    board_error_hist = collections.Counter()

    pbar = tqdm(dataloader, desc="Evaluating", leave=False)

    with torch.no_grad():
        for images, label_rotations in pbar:
            images = images.to(device)
            label_rotations = [[lbl.to(device) for lbl in rots] for rots in label_rotations]

            outputs = model(images)

            matched_labels = []
            for i in range(outputs.size(0)):
                losses = [F.cross_entropy(outputs[i], lbl) for lbl in label_rotations[i]]
                min_idx = torch.argmin(torch.stack(losses))
                total_loss += losses[min_idx].item()
                matched_labels.append(label_rotations[i][min_idx])

            preds = outputs.argmax(dim=2)
            matched_labels_tensor = torch.stack(matched_labels)

            mask = matched_labels_tensor != empty_class_idx
            correct_all += (preds == matched_labels_tensor).sum().item()
            total_all += matched_labels_tensor.numel()
            correct_non_empty += ((preds == matched_labels_tensor) & mask).sum().item()
            total_non_empty += mask.sum().item()

            acc = correct_all / total_all if total_all > 0 else 0.0
            pbar.set_postfix(acc=f"{acc:.2%}")

            batch_errors = (preds != matched_labels_tensor).sum(dim=1)
            for err in batch_errors.cpu().tolist():
                board_error_hist[err] += 1

    avg_loss = total_loss / len(dataloader)
    acc_all = correct_all / total_all if total_all > 0 else 0.0
    acc_non_empty = correct_non_empty / total_non_empty if total_non_empty > 0 else 0.0

    print(f"\n📊 Validation Loss: {avg_loss:.4f}")
    print(f"✅ Accuracy (all squares): {acc_all:.4f}")
    print(f"♟️ Accuracy (non-empty squares only): {acc_non_empty:.4f}")
    print("\n🧮 Board-level prediction accuracy (errors per board):")
    for k in sorted(board_error_hist):
        print(f"  Boards with {k:2d} wrong squares: {board_error_hist[k]}")

    return avg_loss, acc_all, acc_non_empty, board_error_hist


# EVAL FUNCTION (VISUALIZE)

In [ ]:
from tqdm.notebook import tqdm
def evaluate_rotation_invariant_visuals(model, dataloader, device, empty_class_idx=12, image_paths=None, label_to_fen=None):
    import collections
    import matplotlib.pyplot as plt
    import cv2

    model.eval()
    total_loss = 0.0
    correct_all = 0
    correct_non_empty = 0
    total_all = 0
    total_non_empty = 0
    board_error_hist = collections.Counter()

    pbar = tqdm(dataloader, desc="Evaluating", leave=False)
    example_idx = 0  # to track index for image_paths

    with torch.no_grad():
        for images, label_rotations, image_paths_batch in pbar:
            images = images.to(device)
            label_rotations = [[lbl.to(device) for lbl in rots] for rots in label_rotations]

            outputs = model(images)
            matched_labels = []
            for i in range(outputs.size(0)):
                losses = [F.cross_entropy(outputs[i], lbl) for lbl in label_rotations[i]]
                min_idx = torch.argmin(torch.stack(losses))
                total_loss += losses[min_idx].item()
                matched_labels.append(label_rotations[i][min_idx])

            preds = outputs.argmax(dim=2)
            matched_labels_tensor = torch.stack(matched_labels)

            mask = matched_labels_tensor != empty_class_idx
            correct_all += (preds == matched_labels_tensor).sum().item()
            total_all += matched_labels_tensor.numel()
            correct_non_empty += ((preds == matched_labels_tensor) & mask).sum().item()
            acc = correct_all / total_all if total_all > 0 else 0.0
            pbar.set_postfix(acc=f"{acc:.2%}")

            batch_errors = (preds != matched_labels_tensor).sum(dim=1)

            for i in range(len(batch_errors)):
                num_errors = batch_errors[i].item()
                board_error_hist[num_errors] += 1

                if num_errors == 0 and image_paths is not None and label_to_fen is not None:
                    image_path = image_paths_batch[i]

                    img = cv2.imread(image_path)
                    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

                    gt_fen = label_to_fen(matched_labels_tensor[i].cpu())
                    pred_fen = label_to_fen(preds[i].cpu())

                    # --- Plot image with FENs ---
                    plt.figure(figsize=(6, 6))
                    plt.imshow(img)
                    plt.axis("off")
                    plt.title("✅ Correct Prediction")
                    plt.figtext(0.5, -0.1, f"GT FEN:   {gt_fen}\nPred FEN: {pred_fen}",
                                wrap=True, horizontalalignment='center', fontsize=10)
                    plt.tight_layout()
                    plt.show()

                example_idx += 1

    avg_loss = total_loss / len(dataloader)
    acc_all = correct_all / total_all
    acc_non_empty = correct_non_empty / total_non_empty

    print(f"\n📊 Validation Loss: {avg_loss:.4f}")
    print(f"✅ Accuracy (all squares): {acc_all:.4f}")
    print(f"♟️ Accuracy (non-empty squares only): {acc_non_empty:.4f}")
    print("\n🧮 Board-level prediction accuracy (errors per board):")
    for k in sorted(board_error_hist):
        print(f"  Boards with {k:2d} wrong squares: {board_error_hist[k]}")

    return avg_loss, acc_all, acc_non_empty, board_error_hist

# EVAL FUNCTION (VISUALIZE MORE)

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt

# --- minimal defaults so this runs standalone; override via args if you already have these ---
DEFAULT_ID_TO_PIECE = {
    0:'P',1:'R',2:'N',3:'B',4:'Q',5:'K', 6:'p',7:'r',8:'n',9:'b',10:'q',11:'k', 12:'1'
}

def _basic_legality_checks(pred_labels_64, id_to_piece=DEFAULT_ID_TO_PIECE):
    """
    Very light legality: 1 white king, 1 black king; no pawns on ranks 1/8; <=8 pawns/side.
    pred_labels_64: (64,) int tensor or array
    """
    arr = np.asarray(pred_labels_64, dtype=int)
    pieces = [id_to_piece[int(v)] for v in arr]
    # counts
    wK = sum(p=='K' for p in pieces)
    bK = sum(p=='k' for p in pieces)
    wP = sum(p=='P' for p in pieces)
    bP = sum(p=='p' for p in pieces)
    if wK != 1 or bK != 1:
        return False
    # pawn rank rule (ranks from top -> bottom in your labeling)
    top_rank   = pieces[:8]
    bottom_rank= pieces[-8:]
    if any(p=='P' for p in top_rank) or any(p=='P' for p in bottom_rank):
        return False
    if any(p=='p' for p in top_rank) or any(p=='p' for p in bottom_rank):
        return False
    if wP > 8 or bP > 8:
        return False
    return True

def _confusion_and_prf1(gt, pr, num_classes=13):
    """
    gt, pr: 1D int arrays of same length (over all squares)
    Returns: conf_mat (C,C), per-class precision/recall/f1, support
    """
    gt = np.asarray(gt, dtype=int)
    pr = np.asarray(pr, dtype=int)
    conf = np.zeros((num_classes, num_classes), dtype=np.int64)
    for g, p in zip(gt, pr):
        conf[g, p] += 1

    # per-class metrics
    eps = 1e-12
    tp = np.diag(conf)
    fp = conf.sum(axis=0) - tp
    fn = conf.sum(axis=1) - tp
    prec = tp / np.maximum(tp+fp, eps)
    rec  = tp / np.maximum(tp+fn, eps)
    f1   = 2*prec*rec / np.maximum(prec+rec, eps)
    support = conf.sum(axis=1)
    return conf, prec, rec, f1, support

def _ece_reliability(max_probs, correct, n_bins=15):
    """
    max_probs: (N,) float in [0,1]; correct: (N,) bool
    Returns: ece, (bin_conf, bin_acc, bin_count)
    """
    max_probs = np.asarray(max_probs)
    correct = np.asarray(correct).astype(np.float32)
    bins = np.linspace(0, 1, n_bins+1)
    ece = 0.0
    bin_conf = []
    bin_acc  = []
    bin_count= []
    N = len(max_probs)
    for i in range(n_bins):
        lo, hi = bins[i], bins[i+1]
        mask = (max_probs > lo) & (max_probs <= hi) if i>0 else (max_probs >= lo) & (max_probs <= hi)
        cnt = mask.sum()
        if cnt == 0:
            bin_conf.append(np.nan)
            bin_acc.append(np.nan)
            bin_count.append(0)
            continue
        conf = max_probs[mask].mean()
        acc  = correct[mask].mean()
        ece += (cnt / N) * abs(acc - conf)
        bin_conf.append(conf); bin_acc.append(acc); bin_count.append(int(cnt))
    return float(ece), np.array(bin_conf), np.array(bin_acc), np.array(bin_count)

def evaluate_plus(
    model,
    dataloader,
    device,
    empty_class_idx=12,
    id_to_piece=DEFAULT_ID_TO_PIECE,
    compute_plots=True,
    title_prefix="Eval"
):
    """
    Rotation-invariant evaluation with rich metrics:
      - exact-board accuracy, error histogram + CDF, mean/median hamming
      - square acc (all / non-empty), top-2 acc
      - per-class precision/recall/F1, confusion matrix
      - calibration (ECE + reliability curve)
      - 8x8 correctness heatmap (avg over dataset), light/dark split
      - legality pass rate on predicted boards
      - rotation choice distribution (0°,90°,180°,270°)
    Returns: dict of metrics, plus (optionally) plots.
    """
    model.eval()
    total_loss = 0.0
    all_gt = []
    all_pr = []
    all_pr_top2_hits = []
    all_maxprob = []
    all_correct_flags = []

    board_errors = []
    rotation_picks = []  # which rotation idx chosen per board
    per_board_legality = []

    # 8x8 correctness heatmap sums
    heat_sum = np.zeros((8,8), dtype=np.float64)
    heat_cnt = 0

    with torch.no_grad():
        pbar = tqdm(dataloader, desc="evaluate_plus", leave=False)
        for batch in pbar:
            # dataloader returns (images, label_rotations) OR (images, label_rotations, paths)
            if len(batch) == 2:
                images, label_rotations = batch
            else:
                images, label_rotations, _ = batch

            images = images.to(device)
            label_rotations = [[lbl.to(device) for lbl in rots] for rots in label_rotations]

            outputs = model(images)  # (B, 64, 13)
            # softmax for probs
            probs = F.softmax(outputs, dim=2)
            max_prob, pred = probs.max(dim=2)  # (B,64)

            # top-2
            top2 = probs.topk(k=2, dim=2).indices  # (B,64,2)

            # choose best rotation per sample using CE loss
            B = outputs.size(0)
            matched_labels = []
            chosen_rot_idx = []
            losses = []
            for i in range(B):
                lossi = [F.cross_entropy(outputs[i], lbl) for lbl in label_rotations[i]]
                li = torch.stack(lossi)
                min_idx = int(torch.argmin(li).item())
                chosen_rot_idx.append(min_idx)
                losses.append(float(lossi[min_idx].item()))
                matched_labels.append(label_rotations[i][min_idx])

            total_loss += np.mean(losses)
            rotation_picks.extend(chosen_rot_idx)

            matched_labels = torch.stack(matched_labels)  # (B,64)
            # accumulate metrics
            gt_np = matched_labels.detach().cpu().numpy().astype(int)
            pr_np = pred.detach().cpu().numpy().astype(int)
            maxp_np = max_prob.detach().cpu().numpy()

            # per-square correctness
            correct_np = (gt_np == pr_np)

            # top-2 hits mask
            top2_np = top2.detach().cpu().numpy()
            pr_top2_hit = np.any(top2_np == gt_np[...,None], axis=2)

            # flatten accumulators
            all_gt.append(gt_np.reshape(-1))
            all_pr.append(pr_np.reshape(-1))
            all_pr_top2_hits.append(pr_top2_hit.reshape(-1))
            all_maxprob.append(maxp_np.reshape(-1))
            all_correct_flags.append(correct_np.reshape(-1))

            # board-level
            board_err = (gt_np != pr_np).sum(axis=1)  # per board hamming
            board_errors.extend(board_err.tolist())

            # legality per board
            for i in range(B):
                per_board_legality.append(_basic_legality_checks(pr_np[i], id_to_piece=id_to_piece))

            # heatmap accum
            for i in range(B):
                hm = correct_np[i].reshape(8,8).astype(np.float32)
                heat_sum += hm
                heat_cnt += 1

    # concat everything
    all_gt = np.concatenate(all_gt, axis=0)
    all_pr = np.concatenate(all_pr, axis=0)
    all_pr_top2_hits = np.concatenate(all_pr_top2_hits, axis=0)
    all_maxprob = np.concatenate(all_maxprob, axis=0)
    all_correct_flags = np.concatenate(all_correct_flags, axis=0)

    # loss average
    avg_loss = total_loss / max(1, len(dataloader))

    # square-level acc
    acc_all = (all_gt == all_pr).mean()
    non_empty_mask = (all_gt != empty_class_idx)
    acc_non_empty = (all_gt[non_empty_mask] == all_pr[non_empty_mask]).mean() if non_empty_mask.any() else float('nan')

    # top-2 acc
    top2_all = all_pr_top2_hits.mean()
    top2_non_empty = all_pr_top2_hits[non_empty_mask].mean() if non_empty_mask.any() else float('nan')

    # confusion + PRF1 (all classes)
    conf, prec, rec, f1, support = _confusion_and_prf1(all_gt, all_pr, num_classes=13)

    # exclude empty for macro over pieces
    piece_classes = [i for i in range(13) if i != empty_class_idx]
    macro_f1_pieces = np.nanmean(f1[piece_classes])
    macro_prec_pieces = np.nanmean(prec[piece_classes])
    macro_rec_pieces  = np.nanmean(rec[piece_classes])

    # board-level stats
    board_errors = np.array(board_errors)
    exact_board_acc = np.mean(board_errors == 0)
    mean_hamming = float(board_errors.mean())
    median_hamming = float(np.median(board_errors))
    q25 = float(np.percentile(board_errors, 25))
    q75 = float(np.percentile(board_errors, 75))

    # CDF up to, say, 10 errors for presentation
    cdf_k = list(range(0, 11))
    cdf_vals = [float(np.mean(board_errors <= k)) for k in cdf_k]

    # calibration (ECE)
    ece, bin_conf, bin_acc, bin_count = _ece_reliability(all_maxprob, all_correct_flags, n_bins=15)

    # heatmaps
    heat_avg = heat_sum / max(1, heat_cnt)  # (8,8), fraction correct per square
    # light/dark split: a1 is dark; pattern alternates. Let's assume top-left (row 0, col 0) corresponds to a8 (dark).
    light_mask = np.fromfunction(lambda r,c: ((r+c)%2==1), (8,8))  # light squares
    dark_mask  = ~light_mask
    acc_light = float(heat_avg[light_mask].mean())
    acc_dark  = float(heat_avg[dark_mask].mean())

    # legality rate
    legality_rate = float(np.mean(per_board_legality))

    # rotation distribution
    rotation_picks = np.array(rotation_picks)
    rot_hist = {0:int(np.sum(rotation_picks==0)),
                1:int(np.sum(rotation_picks==1)),
                2:int(np.sum(rotation_picks==2)),
                3:int(np.sum(rotation_picks==3))}
    rot_dist = {k: v/len(rotation_picks) for k,v in rot_hist.items()}

    # --- package results ---
    results = {
        "avg_loss": float(avg_loss),

        "square_acc_all": float(acc_all),
        "square_acc_non_empty": float(acc_non_empty),
        "top2_acc_all": float(top2_all),
        "top2_acc_non_empty": float(top2_non_empty),

        "confusion_matrix": conf,                 # np.array (13,13)
        "per_class_precision": prec,              # np.array (13,)
        "per_class_recall": rec,                  # np.array (13,)
        "per_class_f1": f1,                       # np.array (13,)
        "per_class_support": support,             # np.array (13,)
        "macro_prec_pieces": float(macro_prec_pieces),
        "macro_rec_pieces": float(macro_rec_pieces),
        "macro_f1_pieces": float(macro_f1_pieces),

        "exact_board_acc": float(exact_board_acc),
        "board_error_hist": {int(k): int(v) for k,v in zip(*np.unique(board_errors, return_counts=True))},
        "board_error_mean": mean_hamming,
        "board_error_median": median_hamming,
        "board_error_q25": q25,
        "board_error_q75": q75,
        "board_error_cdf_k": cdf_k,
        "board_error_cdf_vals": cdf_vals,

        "ece": float(ece),
        "reliability_bin_conf": bin_conf,
        "reliability_bin_acc": bin_acc,
        "reliability_bin_count": bin_count,

        "heatmap_8x8": heat_avg,                  # np.array (8,8), fraction correct
        "acc_light_squares": acc_light,
        "acc_dark_squares": acc_dark,

        "legality_rate": legality_rate,

        "rotation_hist": rot_hist,
        "rotation_dist": rot_dist,
    }

    if compute_plots:
        # 1) Board error histogram (clipped for readability)
        be = np.clip(board_errors, 0, 15)
        plt.figure(figsize=(6,4))
        plt.hist(be, bins=np.arange(17)-0.5, rwidth=0.9)
        plt.title(f"{title_prefix}: Board Errors (clipped at 15)")
        plt.xlabel("# wrong squares"); plt.ylabel("count")
        plt.show()

        # 2) CDF
        plt.figure(figsize=(6,4))
        plt.plot(cdf_k, cdf_vals, marker='o')
        plt.title(f"{title_prefix}: CDF of Board Errors")
        plt.xlabel("≤ errors"); plt.ylabel("fraction of boards")
        plt.grid(True, alpha=0.3); plt.show()

        # 3) Confusion matrix (small, readable)
        plt.figure(figsize=(7,6))
        plt.imshow(conf, interpolation='nearest')
        plt.title(f"{title_prefix}: Confusion Matrix (13x13)")
        plt.xlabel("Predicted"); plt.ylabel("Ground Truth")
        plt.colorbar(); plt.tight_layout(); plt.show()

        # 4) Reliability curve
        ok = ~np.isnan(bin_conf) & ~np.isnan(bin_acc)
        plt.figure(figsize=(6,4))
        plt.plot(bin_conf[ok], bin_acc[ok], marker='o', label='empirical')
        plt.plot([0,1],[0,1], linestyle='--', label='ideal')
        sizes = bin_count[ok]
        plt.title(f"{title_prefix}: Reliability (ECE={ece:.3f})")
        plt.xlabel("confidence"); plt.ylabel("accuracy")
        plt.legend(); plt.grid(True, alpha=0.3); plt.show()

        # 5) 8x8 heatmap
        plt.figure(figsize=(5,5))
        plt.imshow(heat_avg, vmin=0, vmax=1)
        plt.title(f"{title_prefix}: Square Accuracy Heatmap")
        plt.colorbar(label="fraction correct")
        plt.xticks(range(8)); plt.yticks(range(8))
        plt.tight_layout(); plt.show()

        # 6) Rotation histogram
        plt.figure(figsize=(5,3))
        xs = [0,1,2,3]
        vals = [rot_hist[k] for k in xs]
        plt.bar(xs, vals)
        plt.xticks(xs, ["0°","90°","180°","270°"])
        plt.title(f"{title_prefix}: Chosen Rotation Counts")
        plt.tight_layout(); plt.show()

    return results


# RESNET (PRETRAINED MODEL)

In [ ]:
# PRETRAINED MODEL, use to verify that this is even possible

import torch
import torch.nn as nn
import torchvision.models as models

class ChessPieceClassifier(nn.Module):
    def __init__(self, num_classes=12):
        super(ChessPieceClassifier, self).__init__()
        resnet = models.resnet18(pretrained=True)

        # Remove ResNet classification head
        self.features = nn.Sequential(*list(resnet.children())[:-2])  # Output: (512, 8, 8)

        self.head = nn.Sequential(
            nn.Conv2d(512, 128, kernel_size=1),
            nn.ReLU(),
            nn.Conv2d(128, 13, kernel_size=1)  # 13 classes including empty
        )



    def forward(self, x):
        x = self.features(x)
        x = self.head(x)  # (B, 12, 8, 8)
        x = x.permute(0, 2, 3, 1).reshape(x.size(0), 64, 13)
        return x

import torch.optim as optim
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = ChessPieceClassifier(num_classes=13)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)


In [ ]:
model_overfit = ChessPieceClassifier(num_classes=13).to(device)
optimizer = torch.optim.Adam(model_overfit.parameters(), lr=1e-4)
criterion = torch.nn.CrossEntropyLoss(ignore_index=12)

# Get one batch from your dataloader
images, labels = next(iter(train_loader))

images = images.to(device)
labels = labels.long().to(device)

model_overfit.train()

print("Starting overfit test on single batch")

for i in range(100):  # train 100 iterations on same batch
    optimizer.zero_grad()
    outputs = model_overfit(images)  # (B, 64, 13)

    loss = criterion(outputs.reshape(-1, 13), labels.reshape(-1))
    loss.backward()
    optimizer.step()

    if (i+1) % 10 == 0:
        print(f"Iter {i+1:03d}: Loss = {loss.item():.4f}")


In [ ]:
import torch.nn.functional as F
if torch.cuda.is_available():
    device = 'cuda'
else:
    device = 'cpu'

import torch
import torch.nn as nn
import torchvision.models as models
def per_class_accuracy(preds, labels, num_classes=13):
    """
    preds: (B, 64) tensor of predicted class indices
    labels: (B, 64) tensor of ground truth class indices
    """
    correct = torch.zeros(num_classes, dtype=torch.int32)
    total = torch.zeros(num_classes, dtype=torch.int32)

    for cls in range(num_classes):
        mask = labels == cls
        correct[cls] = (preds[mask] == cls).sum()
        total[cls] = mask.sum()

    acc = correct.float() / total.clamp(min=1).float()
    return acc, correct, total

class ChessPieceClassifier(nn.Module):
    def __init__(self, num_classes=12):
        super(ChessPieceClassifier, self).__init__()
        resnet = models.resnet18(pretrained=True)

        # Remove ResNet classification head
        self.features = nn.Sequential(*list(resnet.children())[:-2])  # Output: (512, 8, 8)

        self.head = nn.Sequential(
            nn.Conv2d(512, 128, kernel_size=1),
            nn.ReLU(),
            nn.Conv2d(128, 13, kernel_size=1)  # 13 classes including empty
        )



    def forward(self, x):
        x = self.features(x)
        x = self.head(x)  # (B, 12, 8, 8)
        x = x.permute(0, 2, 3, 1).reshape(x.size(0), 64, 13)
        return x
def compute_min_rotation_loss(outputs, label_rotations):
    """
    outputs: (B, 64, 13)
    label_rotations: list of B elements, each a list of 4 (64,) tensors
    """
    total_loss = 0
    for i in range(outputs.size(0)):
        losses = []
        for rotated_label in label_rotations[i]:
            loss = F.cross_entropy(outputs[i], rotated_label.to(outputs.device))
            losses.append(loss)
        total_loss += torch.min(torch.stack(losses))
    return total_loss / outputs.size(0)



# UNCOMMENT TO SEE OVERFITTING

# # Get one batch from your dataloader
# images, label_rotations = next(iter(train_loader))

# images = images.to(device)
# label_rotations = [[lbl.to(device) for lbl in rotations] for rotations in label_rotations]

# model_overfit = ChessPieceClassifier(num_classes=13).to(device)
# optimizer = torch.optim.Adam(model_overfit.parameters(), lr=1e-4)

# model_overfit.train()

# print("Starting overfit test on single batch (rotation-invariant)")

# for step in range(100):  # train 100 iterations on same batch
#     optimizer.zero_grad()
#     outputs = model_overfit(images)  # (B, 64, 13)

#     loss = compute_min_rotation_loss(outputs, label_rotations)
#     loss.backward()
#     optimizer.step()

#     # Evaluate rotation-matched labels
#     with torch.no_grad():
#         preds = outputs.argmax(dim=-1)  # (B, 64)

#         # Match lowest-loss label for each image
#         min_losses = []
#         for j in range(images.size(0)):
#             losses = [F.cross_entropy(outputs[j], lbl, ignore_index=12) for lbl in label_rotations[j]]
#             min_index = torch.argmin(torch.stack(losses))
#             min_losses.append(label_rotations[j][min_index])

#         matched_labels = torch.stack(min_losses)  # (B, 64)
#         acc, correct, total = per_class_accuracy(preds, matched_labels)

#     if (step + 1) % 10 == 0:
#         print(f"Iter {step + 1:03d}: Loss = {loss.item():.4f}")
#         for cls_id, (a, c, t) in enumerate(zip(acc, correct, total)):
#             print(f"  Class {cls_id:2d}: Acc = {a:.2%} ({c}/{t})")




In [ ]:
model = ChessPieceClassifier(num_classes=13)
model.to(device)

train(
    model=model,
    dataloader=train_loader,
    device=device,
    epochs=15,
    save_model=True,
    save_dir="/content/drive/MyDrive/Colab Notebooks/APS360/Project/rot-inv-chessred-full",
    empty_class_weight=0.1,
    val_loader=val_loader  # <-- include this
)


In [ ]:
evaluate_rotation_invariant(model, test_loader, device=device)

# CUSTOM MODEL #1 - customV3-chessred-full

In [ ]:
import torch.nn as nn

class CustomChessCNN(nn.Module):
    def __init__(self, num_classes=13):
        super().__init__()

        self.features = nn.Sequential(
            # Conv Block 1
            nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1),  # 224x224 → 224x224
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),  # → 112x112

            # Conv Block 2
            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),  # → 112x112
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),  # → 56x56

            # Conv Block 3
            nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=1),  # → 56x56
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2),  # → 28x28

            # Conv Block 4
            nn.Conv2d(256, 512, kernel_size=3, stride=1, padding=1),  # → 28x28
            nn.BatchNorm2d(512),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((8, 8))  # → 8x8 output for 64 squares
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),  # → B x (512*8*8)
            nn.Linear(512 * 8 * 8, 1024),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(1024, 64 * num_classes)
        )

        self.num_classes = num_classes

    def forward(self, x):
        x = self.features(x)         # B x 512 x 8 x 8
        x = self.classifier(x)       # B x (64 * 13)
        return x.view(-1, 64, self.num_classes)  # B x 64 x 13


In [ ]:
customModel = CustomChessCNN(num_classes=13)
train(
    model=customModel,
    dataloader=train_loader,
    device=device,
    epochs=15,
    save_model=True,
    save_dir="/content/drive/MyDrive/Colab Notebooks/APS360/Project/customV3-chessred-full",
    empty_class_weight=0.1,
    val_loader=val_loader
)

# CUSTOM MODEL #2 - customV4-residuals-chessred-full

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, downsample=None):
        super(ResidualBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3,
                               stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3,
                               stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)

        self.downsample = downsample  # 1x1 conv if dimensions change

    def forward(self, x):
        identity = x

        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))

        if self.downsample is not None:
            identity = self.downsample(x)

        out += identity
        return self.relu(out)

class CustomChessCNN_v2(nn.Module):
    def __init__(self, num_classes=13):
        super(CustomChessCNN_v2, self).__init__()

        self.in_channels = 64

        self.initial = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False),  # Downsample
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1)  # Downsample
        )

        self.layer1 = self._make_layer(64, 2)
        self.layer2 = self._make_layer(128, 2, stride=2)
        self.layer3 = self._make_layer(256, 2, stride=2)
        self.layer4 = self._make_layer(512, 2, stride=2)

        self.avgpool = nn.AdaptiveAvgPool2d((8, 8))  # Output shape: (B, 512, 8, 8)
        self.classifier = nn.Linear(512, num_classes)  # Apply per square (not globally)


    def _make_layer(self, out_channels, blocks, stride=1):
        downsample = None

        if stride != 1 or self.in_channels != out_channels:
            downsample = nn.Sequential(
                nn.Conv2d(self.in_channels, out_channels,
                          kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

        layers = [ResidualBlock(self.in_channels, out_channels, stride, downsample)]
        self.in_channels = out_channels

        for _ in range(1, blocks):
            layers.append(ResidualBlock(out_channels, out_channels))

        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.initial(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        x = self.avgpool(x)                 # B x 512 x 8 x 8
        x = x.permute(0, 2, 3, 1)           # B x 8 x 8 x 512
        x = x.reshape(x.size(0), 64, 512)   # B x 64 x 512

        x = self.classifier(x)             # B x 64 x 13
        return x


In [ ]:
model = CustomChessCNN_v2(num_classes=13)
model.to(device)

train(
    model=model,
    dataloader=train_loader,
    device=device,
    epochs=35,
    save_model=True,
    save_dir="/content/drive/MyDrive/Colab Notebooks/APS360/Project/customV4-residuals-chessred-full",  # your path
    val_loader=val_loader
)


# CUSTOM MODEL #3 - customV5-residuals-dropout-chessred-full

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class PreActResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, dropout=0.0):
        super().__init__()
        self.bn1 = nn.BatchNorm2d(in_channels)
        self.relu1 = nn.ReLU(inplace=True)
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)

        self.bn2 = nn.BatchNorm2d(out_channels)
        self.relu2 = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False)

        self.downsample = None
        if stride != 1 or in_channels != out_channels:
            self.downsample = nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False)

        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()

    def forward(self, x):
        identity = x

        out = self.relu1(self.bn1(x))
        if self.downsample:
            identity = self.downsample(out)

        out = self.conv1(out)
        out = self.relu2(self.bn2(out))
        out = self.dropout(out)
        out = self.conv2(out)

        return out + identity

class CustomChessCNN_v3(nn.Module):
    def __init__(self, in_channels=3, num_classes=13, dropout=0.3):
        super().__init__()

        self.stem = nn.Sequential(
            nn.Conv2d(in_channels, 64, kernel_size=7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        )

        # Residual layers (increase channels and downsample)
        self.layer1 = self._make_layer(64, 128, blocks=2, stride=1, dropout=dropout)
        self.layer2 = self._make_layer(128, 256, blocks=2, stride=2, dropout=dropout)
        self.layer3 = self._make_layer(256, 512, blocks=2, stride=2, dropout=dropout)

        self.avgpool = nn.AdaptiveAvgPool2d((8, 8))  # output shape: (B, 512, 8, 8)
        self.classifier = nn.Linear(512, num_classes)  # for each square

    def _make_layer(self, in_channels, out_channels, blocks, stride, dropout):
        layers = [PreActResidualBlock(in_channels, out_channels, stride, dropout)]
        for _ in range(1, blocks):
            layers.append(PreActResidualBlock(out_channels, out_channels, dropout=dropout))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)

        x = self.avgpool(x)  # (B, 512, 8, 8)
        x = x.permute(0, 2, 3, 1)  # (B, 8, 8, 512)
        x = x.view(x.size(0), 64, -1)  # (B, 64, 512)
        x = self.classifier(x)  # (B, 64, 13)
        return x


In [ ]:
from tqdm.notebook import tqdm

model = CustomChessCNN_v3(num_classes=13, dropout=0.3).to(device)
train(
    model=model,
    dataloader=train_loader,
    device=device,
    epochs=35,
    save_model=True,
    save_dir="/content/drive/MyDrive/Colab Notebooks/APS360/Project/customV5-residuals-dropout-chessred-full",  # your path
    val_loader=val_loader
)

In [ ]:
model = CustomChessCNN_v3(num_classes=13, dropout=0.3)  # or whatever class you used
model.load_state_dict(torch.load("/content/drive/MyDrive/Colab Notebooks/APS360/Project/customV5-residuals-dropout-chessred-full/epoch35.pth"))
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model.to(device)

total_params = sum(p.numel() for p in model.parameters())

In [ ]:
print(total_params)

In [ ]:
from tqdm.notebook import tqdm
evaluate_rotation_invariant(model, test_loader_plain, device=device)

In [ ]:
res = evaluate_plus(
    model,
    dataloader=test_loader_plain,   # or val_loader
    device=device,
    empty_class_idx=12,
    id_to_piece=id_to_piece,
    compute_plots=True,
    title_prefix="Test"
)

# quick headline print
print({
    "Exact-board acc": res["exact_board_acc"],
    "Square acc (all)": res["square_acc_all"],
    "Square acc (non-empty)": res["square_acc_non_empty"],
    "Macro F1 (pieces)": res["macro_f1_pieces"],
    "Top-2 acc (all)": res["top2_acc_all"],
    "Legality rate": res["legality_rate"],
    "ECE": res["ece"],
})


In [ ]:
print(res)

## VISUALIZE TEST RESULTS

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, PowerNorm

id_to_piece = {
    0: 'P', 1: 'R', 2: 'N', 3: 'B', 4: 'Q', 5: 'K',
    6: 'p', 7: 'r', 8: 'n', 9: 'b', 10: 'q', 11: 'k',
    12: '1'  # empty
}

cm = res["confusion_matrix"].astype(float)
labels = [id_to_piece[i] for i in range(len(id_to_piece))]

# Row-normalized (recall-style)
row_sums = cm.sum(axis=1, keepdims=True)
cm_row = np.divide(cm, np.maximum(row_sums, 1), where=row_sums > 0)

fig = plt.figure(figsize=(6, 5))
im = plt.imshow(cm_row, cmap="magma", norm=PowerNorm(gamma=0.4))  # boost low values
plt.title("Test: Confusion Matrix (row-normalized)")
plt.xlabel("Predicted")
plt.ylabel("Ground Truth")
plt.colorbar(im, fraction=0.046, pad=0.04, label="fraction")
plt.xticks(range(len(labels)), labels)
plt.yticks(range(len(labels)), labels)

# Annotate only reasonably large cells
for i in range(cm_row.shape[0]):
    for j in range(cm_row.shape[1]):
        v = cm_row[i, j]
        if v > 0.01:
            plt.text(j, i, f"{v:.2f}", va="center", ha="center",
                     fontsize=7, color="white")
plt.tight_layout()
plt.show()

# Raw counts with log scale
fig = plt.figure(figsize=(6, 5))
im = plt.imshow(cm + 1, cmap="viridis",
                norm=LogNorm(vmin=1, vmax=cm.max()))  # +1 avoids log(0)
plt.title("Test: Confusion Matrix (counts, log scale)")
plt.xlabel("Predicted")
plt.ylabel("Ground Truth")
plt.colorbar(im, fraction=0.046, pad=0.04, label="count (log)")
plt.xticks(range(len(labels)), labels)
plt.yticks(range(len(labels)), labels)
plt.tight_layout()
plt.show()


In [ ]:
acc8 = np.array(res["heatmap_8x8"], dtype=float)
global_mean = acc8.mean()
vmin = max(acc8.min(), global_mean - 0.01)  # clamp lower bound near mean
vmax = acc8.max()

fig = plt.figure(figsize=(4.6, 4.2))
im = plt.imshow(acc8, cmap="plasma", vmin=vmin, vmax=vmax)
plt.title("Test: Square Accuracy Heatmap (contrast-stretched)")
plt.xlabel("file (a→h)"); plt.ylabel("rank (8→1)")
plt.colorbar(im, fraction=0.046, pad=0.04, label="fraction correct")

# Optional annotations for small boards
for r in range(8):
    for c in range(8):
        plt.text(c, r, f"{acc8[r,c]*100:.1f}%", ha="center", va="center", fontsize=6, color="white")
plt.tight_layout(); plt.show()


In [ ]:
import matplotlib.pyplot as plt
from collections import Counter

err_hist = Counter(res["board_error_hist"])
# Expand to a list if you prefer exact distribution:
# errors = []
# for k, v in err_hist.items(): errors += [k]*v

# Histogram (clip right tail for readability)
clip_at = 15
xs = list(range(0, clip_at+1))
ys = [sum(v for k,v in err_hist.items() if k==x) for x in xs]

plt.figure(figsize=(4.3,3.6))
plt.bar(xs, ys)
plt.yscale("log")  # highlights tails without rerunning
plt.xlabel("# wrong squares"); plt.ylabel("count (log)")
plt.title("Test: Board Errors (log scale)"); plt.tight_layout(); plt.show()

# CDF (already in eval_stats; just plot with markers)
cdf_x = res["board_error_cdf_k"]
cdf_y = res["board_error_cdf_vals"]
plt.figure(figsize=(4.3,3.6))
plt.plot(cdf_x, cdf_y, marker="o")
plt.ylim(0.6, 1.01); plt.grid(True, alpha=0.3)
plt.xlabel("≤ errors"); plt.ylabel("fraction of boards")
plt.title("Test: CDF of Board Errors"); plt.tight_layout(); plt.show()


In [ ]:
conf = np.array(res["reliability_bin_conf"], float)
acc  = np.array(res["reliability_bin_acc"], float)
mask = np.isfinite(conf) & np.isfinite(acc)
ece  = float(res["ece"])

plt.figure(figsize=(4.8,3.8))
plt.plot(conf[mask], acc[mask], marker="o", label="empirical")
plt.plot([0,1],[0,1], "--", label="ideal")
plt.xlabel("confidence"); plt.ylabel("accuracy")
plt.title(f"Test: Reliability (ECE={ece:.3f})")
plt.grid(True, alpha=0.3); plt.legend(); plt.tight_layout(); plt.show()


In [ ]:
rot_hist = res["rotation_hist"]
labels = ["0°","90°","180°","270°"]
vals = [rot_hist.get(i,0) for i in range(4)]

plt.figure(figsize=(4.2,3.4))
plt.bar(labels, vals)
plt.title("Test: Chosen Rotation Counts")
plt.ylabel("count"); plt.tight_layout(); plt.show()


In [ ]:
evaluate_rotation_invariant_visuals(
    model,
    test_loader,
    device=device,
    image_paths=X_loaded,
    label_to_fen=label_vector_to_fen
)


In [ ]:
import os, cv2, torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

def evaluate_rotation_invariant_visuals_plus(
    model,
    dataloader,
    device,
    empty_class_idx=12,
    image_paths=None,            # list of paths aligned with the dataset; if dataloader returns paths, you can leave this None
    label_to_fen=None,           # callable: tensor(64,) -> FEN string
    preprocess_warp_fn=None,     # callable: (path) -> (warped_img, squares) or None; if None, assumes path already points to the warped image
    max_show_per_bucket=6,       # how many examples to show for each bucket
    high_error_threshold=8,      # "lots of errors" threshold
    save_figs_dir=None           # optional dir to save figures instead of only showing
):
    """
    Runs eval (rotation-invariant matching like your original) and then visualizes:
      - a few 0-error boards
      - a few high-error boards (>= high_error_threshold)
    For each example: show [Input image | Warped (if available)] + GT FEN + Pred FEN.
    """
    import collections

    model.eval()
    total_loss = 0.0
    correct_all = 0
    correct_non_empty = 0
    total_all = 0
    total_non_empty = 0
    board_error_hist = collections.Counter()

    # Keep candidates for qualitative display
    zero_error_examples = []      # list of dicts
    high_error_examples = []      # list of dicts

    os.makedirs(save_figs_dir, exist_ok=True) if save_figs_dir else None

    pbar = tqdm(dataloader, desc="Evaluating (visuals+)", leave=False)

    with torch.no_grad():
        for batch in pbar:
            # dataloader might be (images, label_rotations) OR (images, label_rotations, paths)
            if len(batch) == 3:
                images, label_rotations, paths_batch = batch
            else:
                images, label_rotations = batch
                paths_batch = [None] * images.size(0) if image_paths is None else image_paths

            images = images.to(device)
            label_rotations = [[lbl.to(device) for lbl in rots] for rots in label_rotations]

            outputs = model(images)  # (B, 64, C)
            matched_labels = []
            for i in range(outputs.size(0)):
                losses = [F.cross_entropy(outputs[i], lbl) for lbl in label_rotations[i]]
                min_idx = torch.argmin(torch.stack(losses))
                total_loss += losses[min_idx].item()
                matched_labels.append(label_rotations[i][min_idx])

            preds = outputs.argmax(dim=2)                    # (B, 64)
            matched_labels_tensor = torch.stack(matched_labels)  # (B, 64)

            mask = matched_labels_tensor != empty_class_idx
            correct_all += (preds == matched_labels_tensor).sum().item()
            total_all += matched_labels_tensor.numel()
            correct_non_empty += ((preds == matched_labels_tensor) & mask).sum().item()
            total_non_empty += mask.sum().item()

            acc = correct_all / total_all if total_all > 0 else 0.0
            pbar.set_postfix(acc=f"{acc:.2%}")

            # per-board errors and stash examples
            batch_errors = (preds != matched_labels_tensor).sum(dim=1)
            for i in range(len(batch_errors)):
                num_err = int(batch_errors[i].item())
                board_error_hist[num_err] += 1

                # Build qualitative record (only if we still need more examples)
                want_zero = (num_err == 0 and len(zero_error_examples) < max_show_per_bucket)
                want_high = (num_err >= high_error_threshold and len(high_error_examples) < max_show_per_bucket)

                if (want_zero or want_high) and label_to_fen is not None:
                    path_i = paths_batch[i] if paths_batch is not None else None

                    # Load "input image"
                    input_img = None
                    if path_i and os.path.exists(path_i):
                        bgr = cv2.imread(path_i)
                        if bgr is not None:
                            input_img = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

                    # Warped image:
                    warped_img = None
                    if preprocess_warp_fn is not None and path_i and os.path.exists(path_i):
                        try:
                            warped, _ = preprocess_warp_fn(path_i, display=False)
                            if warped is not None:
                                warped_img = cv2.cvtColor(warped, cv2.COLOR_BGR2RGB)
                        except TypeError:
                            # If preprocess_warp_fn only accepts (path) without kwargs
                            warped = preprocess_warp_fn(path_i)
                            if isinstance(warped, tuple):
                                warped = warped[0]
                            if warped is not None:
                                warped_img = cv2.cvtColor(warped, cv2.COLOR_BGR2RGB)

                    # Fallback: if no preprocess provided, assume the file is already warped
                    if warped_img is None and input_img is not None and preprocess_warp_fn is None:
                        warped_img = input_img

                    rec = {
                        "path": path_i,
                        "num_errors": num_err,
                        "gt_fen": label_to_fen(matched_labels_tensor[i].cpu()),
                        "pred_fen": label_to_fen(preds[i].cpu()),
                        "input_img": input_img,
                        "warped_img": warped_img
                    }

                    if want_zero:
                        zero_error_examples.append(rec)
                    elif want_high:
                        high_error_examples.append(rec)

    # Aggregate metrics
    avg_loss = total_loss / len(dataloader)
    acc_all = correct_all / total_all if total_all > 0 else 0.0
    acc_non_empty = correct_non_empty / total_non_empty if total_non_empty > 0 else 0.0

    print(f"\n📊 Validation Loss: {avg_loss:.4f}")
    print(f"✅ Accuracy (all squares): {acc_all:.4f}")
    print(f"♟️ Accuracy (non-empty squares only): {acc_non_empty:.4f}")
    print("\n🧮 Board-level prediction accuracy (errors per board):")
    for k in sorted(board_error_hist):
        print(f"  Boards with {k:2d} wrong squares: {board_error_hist[k]}")

    # ---------- VISUALIZATION ----------
    def _show_example(rec, title_prefix):
        # Make a 1x2 figure: [Input | Warped]
        fig, axes = plt.subplots(1, 2, figsize=(10, 5))
        ax_in, ax_warp = axes

        if rec["input_img"] is not None:
            ax_in.imshow(rec["input_img"])
            ax_in.set_title("Input image")
        else:
            ax_in.imshow(np.zeros((10,10)))
            ax_in.set_title("Input image (not found)")
        ax_in.axis("off")

        if rec["warped_img"] is not None:
            ax_warp.imshow(rec["warped_img"])
            ax_warp.set_title("Warped image")
        else:
            ax_warp.imshow(np.zeros((10,10)))
            ax_warp.set_title("Warped image (not available)")
        ax_warp.axis("off")

        # FEN text under the figure
        gt = rec["gt_fen"]; pr = rec["pred_fen"]
        nerr = rec["num_errors"]
        path = rec["path"] if rec["path"] else "N/A"
        fig.suptitle(f"{title_prefix}  |  errors={nerr}  |  file={os.path.basename(path)}", fontsize=12)
        plt.figtext(0.5, -0.02, f"GT FEN:   {gt}\nPred FEN: {pr}",
                    ha="center", va="top", wrap=True, fontsize=10)

        plt.tight_layout()
        if save_figs_dir:
            base = os.path.splitext(os.path.basename(path if path else f"{title_prefix}_{nerr}"))[0]
            out = os.path.join(save_figs_dir, f"{title_prefix}_{base}_e{nerr}.png")
            plt.savefig(out, bbox_inches="tight", dpi=150)
        plt.show()

    # Show a few perfect boards
    for rec in zero_error_examples:
        _show_example(rec, "Correct Prediction")

    # Show a few high-error boards
    for rec in high_error_examples:
        _show_example(rec, "High-Error Prediction")

    return {
        "avg_loss": avg_loss,
        "acc_all": acc_all,
        "acc_non_empty": acc_non_empty,
        "board_error_hist": board_error_hist,
        "shown_zero_error": len(zero_error_examples),
        "shown_high_error": len(high_error_examples)
    }
_ = evaluate_rotation_invariant_visuals_plus(
    model,
    test_loader,                  # or test_loader_plain if it has paths too
    device=device,
    label_to_fen=label_vector_to_fen,
    preprocess_warp_fn=None,      # or preprocess_chessboard_real if paths are original
    max_show_per_bucket=6,
    high_error_threshold=8,
)


In [ ]:
import os, cv2, torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms

def _to_standard_fen(fen_str: str) -> str:
    """Keep only the piece-layout field and compress runs of '0' into digits."""
    layout = fen_str.split()[0]  # drop side-to-move/castling if present
    rows_out = []
    for row in layout.split('/'):
        out, zeros = [], 0
        for ch in row:
            if ch == '0':
                zeros += 1
            elif ch.isdigit():            # already-compressed empties
                if zeros: out.append(str(zeros)); zeros = 0
                out.append(ch)
            else:                          # piece letter
                if zeros: out.append(str(zeros)); zeros = 0
                out.append(ch)
        if zeros: out.append(str(zeros))
        rows_out.append(''.join(out))
    return '/'.join(rows_out)

@torch.no_grad()
def evaluate_visuals_from_paths(
    model,
    device,
    warped_paths,                 # list[str] -> model inputs (already warped)
    original_paths=None,          # list[str] or None
    gt_fens=None,                 # list[str] or None
    label_to_fen=None,            # fn: tensor[64] -> FEN (required)
    fen_to_label_vector_fn=None,  # fn: fen -> tensor[64] (optional, for error count)
    empty_class_idx=12,
    resize_hw=(256, 256),
    save_figs_dir=None
):
    assert label_to_fen is not None, "Please pass label_to_fen=label_vector_to_fen"

    # simple eval transform
    tfm = transforms.Compose([transforms.Resize(resize_hw), transforms.ToTensor()])

    model.eval()
    model.to(device)

    # Load warped images
    warped_imgs_rgb, warped_tensors = [], []
    for p in warped_paths:
        img = Image.open(p).convert('RGB')
        warped_imgs_rgb.append(np.array(img))
        warped_tensors.append(tfm(img))
    if not warped_tensors:
        print("No warped images provided.")
        return

    batch = torch.stack(warped_tensors, dim=0).to(device)  # (B,3,H,W)
    logits = model(batch)                                  # (B,64,13)
    preds = logits.argmax(dim=2).cpu()                     # (B,64)

    # Convert predictions to standardized FEN
    pred_fens = [_to_standard_fen(label_to_fen(preds[i])) for i in range(preds.size(0))]

    # Standardize GT FENs if provided
    if gt_fens is None:
        gt_fens_std = [None] * len(warped_paths)
    else:
        gt_fens_std = [_to_standard_fen(f) if f is not None else None for f in gt_fens]

    # Safe original paths list
    if original_paths is None:
        original_paths = [None] * len(warped_paths)

    if save_figs_dir:
        os.makedirs(save_figs_dir, exist_ok=True)

    for i, (wpath, opath, pred_fen, gt_fen_std) in enumerate(
        zip(warped_paths, original_paths, pred_fens, gt_fens_std)
    ):
        # Load original (if provided)
        orig_rgb = None
        if opath and os.path.exists(opath):
            bgr = cv2.imread(opath)
            if bgr is not None:
                orig_rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

        # Compute #errors if possible
        num_err = None
        if gt_fen_std is not None and fen_to_label_vector_fn is not None:
            try:
                gt_vec = fen_to_label_vector_fn(gt_fen_std, empty_char='0')
                num_err = int((preds[i] != gt_vec).sum().item())
            except Exception:
                num_err = None  # be robust if a parsing error happens

        # --- Render panel ---
        fig, axes = plt.subplots(1, 2, figsize=(5,2.5))
        ax0, ax1 = axes

        if orig_rgb is not None:
            ax0.imshow(orig_rgb); ax0.set_title("Input (original)")
        else:
            ax0.imshow(np.zeros((10, 10))); ax0.set_title("Original unavailable")
        ax0.axis("off")

        ax1.imshow(warped_imgs_rgb[i]); ax1.set_title("Warped (model input)")
        ax1.axis("off")

        # Caption (bigger font)
        gt_txt = gt_fen_std if gt_fen_std is not None else "(not provided)"
        err_txt = f"   \n   # wrong squares: {num_err}" if num_err is not None else ""
        plt.suptitle(os.path.basename(wpath), fontsize=14)
        plt.figtext(
            0.5, -0.05,
            f"GT FEN:   {gt_txt}\nPred FEN: {pred_fen}{err_txt}",
            ha="center", va="top", wrap=True, fontsize=10
        )
        plt.tight_layout()

        if save_figs_dir:
            base = os.path.splitext(os.path.basename(wpath))[0]
            out = os.path.join(save_figs_dir, f"qual_{base}.png")
            plt.savefig(out, bbox_inches="tight", dpi=150)

        plt.show()
results = evaluate_visuals_from_paths(
    model,
    device=device,
    warped_paths=["/content/drive/MyDrive/Colab Notebooks/APS360/Project/ChessReD_Hough/28.png"],
    original_paths=["/content/drive/MyDrive/Colab Notebooks/APS360/Project/ChessReD/images/0/G000_IMG028.jpg"],
    gt_fens=[pieces_to_fen(image_to_pieces[annotations['splits']['chessred2k']['train']['image_ids'][28]])],  # or a list like ["rnbqkbnr/..."]
    label_to_fen=label_vector_to_fen,
    fen_to_label_vector_fn=fen_to_label_vector
)

# evaluate_visuals_from_paths(
#     model,
#     device,
#     warped_paths,                 # list[str] -> model inputs (already warped)
#     original_paths=None,          # list[str] or None
#     gt_fens=None,                 # list[str] or None
#     label_to_fen=None,            # fn: tensor[64] -> FEN (required)
#     fen_to_label_vector_fn=None,  # fn: fen -> tensor[64] (optional, for error count)
#     empty_class_idx=12,
#     resize_hw=(256, 256),
#     save_figs_dir=None
# ):

In [ ]:
import cv2, numpy as np, matplotlib.pyplot as plt, torch
import torch.nn.functional as F
from PIL import Image
from torchvision import transforms

def preprocess_chessboard_with_pred(
    image_path,
    gt_fen,                         # e.g. "rnbqkbnr/pppppppp/8/..."; side/castling allowed -> we'll strip
    model,
    device,
    label_to_fen,                   # your label_vector_to_fen
    fen_to_label_vector,            # your fen_to_label_vector
    output_size=400,
    display=True,
    transform=None,
    rotation_invariant=True,
):
    """
    Shows: Original | Dilated edges | Corner overlay | Warped top-down
    and writes GT FEN + Pred FEN under the figure.

    Returns: warped_img (np.ndarray), squares (list of tiles), pred_fen (str), best_rot (0/1/2/3), num_err (int)
    """
    # ---- read + find board corners ----
    img = cv2.imread(image_path)
    if img is None:
        print("Could not read:", image_path); return None, None, None, None, None

    corners, debug_dilated = detect_board_corners(img)
    if corners is None:
        print("no corners")
        return None, None, None, None, None

    corner_overlay = draw_corners(img, corners)
    warped = warp_board(img, corners, output_size)

    # ---- run model on the warped image ----
    if transform is None:
        transform = transforms.Compose([transforms.Resize((256, 256)), transforms.ToTensor()])

    pil_warp = Image.fromarray(cv2.cvtColor(warped, cv2.COLOR_BGR2RGB))
    x = transform(pil_warp).unsqueeze(0).to(device)        # (1,3,H,W)

    model.eval()
    with torch.no_grad():
        logits = model(x)                                  # (1, 64, 13)
        pred_vec = logits.argmax(dim=2)[0].cpu()           # (64,)
        pred_fen = label_to_fen(pred_vec)

    # ---- rotation-invariant comparison to GT (optional) ----
    gt_board = gt_fen.split()[0] if gt_fen is not None else None
    best_rot, num_err = None, None
    if rotation_invariant and gt_board is not None:
        gt_rots = [
            gt_board,
            rotate_fen_90(gt_board),
            rotate_fen_180(gt_board),
            rotate_fen_270(gt_board),
        ]
        errs = []
        for r, rf in enumerate(gt_rots):
            gt_vec = fen_to_label_vector(rf)
            err = int((pred_vec != gt_vec).sum().item())
            errs.append((err, r))
        errs.sort()
        num_err, best_rot = errs[0]

    # ---- show figure (same 4 panels) + FEN text ----
    if display:
        titles = ["Original", "Dilated Edges", "Corner Overlay", "Warped Top-Down"]
        images = [img, debug_dilated, corner_overlay, warped]

        plt.figure(figsize=(60, 20))
        for i in range(4):
            plt.subplot(1, 4, i + 1)
            im = images[i]
            if im.ndim == 2:
                plt.imshow(im, cmap='gray')
            else:
                plt.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
            plt.title(titles[i]); plt.axis('off')

        gt_line = f"GT FEN:   {gt_board if gt_board is not None else '—'}"
        pred_line = f"Pred FEN: {pred_fen}"
        extra = ""
        if num_err is not None:
            extra = f"   |   min errors={num_err} (best rotation={best_rot*90}°)"
        plt.figtext(0.5, 0.02, gt_line + "\n" + pred_line + extra,
                    ha="center", va="bottom", wrap=True, fontsize=12)
        plt.figtext(
            0.5, -0.05,  # moved slightly further down
            gt_line + "\n" + pred_line + extra,
            ha="center", va="top", wrap=True,
            fontsize=28  # much larger font
        )
        plt.tight_layout(); plt.show()

    # tiles if you still want them
    squares = []
    sq = output_size // 8
    for r in range(8):
        for c in range(8):
            squares.append(warped[r*sq:(r+1)*sq, c*sq:(c+1)*sq])

    return warped, squares, pred_fen, best_rot, num_err
warped, squares, pred_fen, best_rot, num_err = preprocess_chessboard_with_pred(
    image_path=image_id_to_path[annotations['splits']['chessred2k']['train']['image_ids'][1]],
    gt_fen=pieces_to_fen(image_to_pieces[annotations['splits']['chessred2k']['train']['image_ids'][1]]) + " w KQkq -",
    model=model, device=device,
    label_to_fen=label_vector_to_fen,
    fen_to_label_vector=fen_to_label_vector,
    output_size=400, display=True
)


# SET UP NEW IMAGES

In [ ]:
import json
import os

real_json_path = "/content/drive/MyDrive/Colab Notebooks/APS360/Project/New Photos/real_labels_2.json"
real_image_dir = "/content/drive/MyDrive/Colab Notebooks/APS360/Project/New Photos/Overhead"

with open(real_json_path, 'r') as f:
    real_data = json.load(f)

image_paths_real = [os.path.join(real_image_dir, item['image']) for item in real_data]
fen_labels_real = [item['gt_fen'] for item in real_data]

print(image_paths_real[2])

In [ ]:
# try preprocessing 1 of the new real images
preprocess_chessboard(image_paths_real[2], display=True)

In [ ]:
preprocess_chessboard("/content/drive/MyDrive/Colab Notebooks/APS360/Project/New Photos/Overhead/IMG_6196.jpg", display=True)

In [ ]:
import cv2, numpy as np, matplotlib.pyplot as plt

# --- HSV green-board mask (wider range + cleanup)
def _green_mask(img_bgr):
    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    # cover dark/medium greens (vinyl boards vary a lot)
    lower = np.array([30, 25, 25], dtype=np.uint8)
    upper = np.array([95, 255, 255], dtype=np.uint8)
    mask = cv2.inRange(hsv, lower, upper)
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k, iterations=2)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN,  k, iterations=1)
    return mask

def order_points_robust_real(pts):
    x_sorted = pts[np.argsort(pts[:, 0]), :]
    left_most = x_sorted[:2, :]
    right_most = x_sorted[2:, :]
    tl, bl = left_most[np.argsort(left_most[:, 1]), :]
    tr, br = right_most[np.argsort(right_most[:, 1]), :]
    return np.array([tl, tr, br, bl], dtype="float32")

def _auto_canny(gray, sigma=0.33):
    v = np.median(gray)
    lo = int(max(0, (1.0 - sigma) * v))
    hi = int(min(255, (1.0 + sigma) * v))
    return cv2.Canny(gray, lo, hi)

def _best_quad_from_contours(contours, image_area, min_area_frac=0.04):
    best, best_score = None, -1.0
    for cnt in contours[:50]:
        peri = cv2.arcLength(cnt, True)
        if peri < 1e-3:
            continue
        approx = cv2.approxPolyDP(cnt, 0.02 * peri, True)
        if len(approx) != 4:
            continue
        area = cv2.contourArea(approx)
        if area < min_area_frac * image_area:
            continue
        pts = order_points_robust_real(approx.reshape(-1, 2).astype(np.float32))
        sides = [np.linalg.norm(pts[i] - pts[(i+1)%4]) for i in range(4)]
        side_ratio = min(sides) / (max(sides) + 1e-6)
        score = (area / image_area) * side_ratio
        if score > best_score:
            best_score, best = score, pts
    return best

# --- NEW: corner detection that masks the table BEFORE edges
def detect_board_corners_real(
    img,
    blur_type="bilateral",   # "gaussian" | "median" | "bilateral" | None
    blur_ksize=5,
    blur_sigma=0,
    bilateral_d=7,
    bilateral_sigma_color=60,
    bilateral_sigma_space=60,
    min_area_frac=0.04
):
    H, W = img.shape[:2]
    image_area = H * W

    # 1) Color mask (green squares)
    gmask = _green_mask(img)
    # If we got a decent mask, fit a rectangle to it to get a board-shaped ROI
    quad_from_rect = None
    if np.count_nonzero(gmask) > 500:
        cnts, _ = cv2.findContours(gmask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if cnts:
            c = max(cnts, key=cv2.contourArea)
            if cv2.contourArea(c) > min_area_frac * image_area * 0.5:
                rect = cv2.minAreaRect(c)
                box = cv2.boxPoints(rect).astype(np.float32)
                quad_from_rect = order_points_robust_real(box)

    # Build an ROI mask (dilate so border/coordinates are inside)
    roi = gmask.copy()
    if np.count_nonzero(roi) > 0:
        roi = cv2.dilate(roi, cv2.getStructuringElement(cv2.MORPH_RECT, (21, 21)), iterations=1)

    # 2) Edge fallback computed ONLY inside ROI
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    if blur_type == "gaussian":
        gray = cv2.GaussianBlur(gray, (blur_ksize, blur_ksize), blur_sigma)
    elif blur_type == "median":
        gray = cv2.medianBlur(gray, blur_ksize)
    elif blur_type == "bilateral":
        gray = cv2.bilateralFilter(gray, bilateral_d, bilateral_sigma_color, bilateral_sigma_space)

    edges = _auto_canny(gray, sigma=0.33)
    if np.count_nonzero(roi) > 0:
        edges = cv2.bitwise_and(edges, roi)  # table edges suppressed
    dilated = cv2.dilate(edges, cv2.getStructuringElement(cv2.MORPH_RECT, (7, 7)), iterations=2)

    contours, _ = cv2.findContours(dilated, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    contours = sorted(contours, key=cv2.contourArea, reverse=True)
    quad_edge = _best_quad_from_contours(contours, image_area, min_area_frac=min_area_frac)

    # Prefer the explicit quad from mask-fitting if it’s reasonable
    if quad_from_rect is not None and quad_edge is None:
        return quad_from_rect, dilated
    if quad_edge is not None and quad_from_rect is None:
        return quad_edge, dilated
    if quad_from_rect is not None and quad_edge is not None:
        # choose the larger area quad
        a1 = cv2.contourArea(quad_from_rect.astype(np.float32))
        a2 = cv2.contourArea(quad_edge.astype(np.float32))
        return (quad_edge if a2 > a1 else quad_from_rect), dilated

    return None, dilated

def draw_corners_real(img, corners):
    img_copy = img.copy()
    labels = ["TL", "TR", "BR", "BL"]
    for i, pt in enumerate(corners):
        p = tuple(np.int32(pt))
        cv2.circle(img_copy, p, 10, (0,0,255), -1)
        cv2.putText(img_copy, labels[i], (p[0]+10, p[1]-10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255,0,0), 2)
    for i in range(4):
        p1 = tuple(np.int32(corners[i])); p2 = tuple(np.int32(corners[(i+1)%4]))
        cv2.line(img_copy, p1, p2, (0,255,0), 2)
    return img_copy

def warp_board_real(img, corners, output_size=400):
    src = order_points_robust_real(corners)
    dst = np.array([[0,0],[output_size-1,0],[output_size-1,output_size-1],[0,output_size-1]], dtype="float32")
    M = cv2.getPerspectiveTransform(src, dst)
    return cv2.warpPerspective(img, M, (output_size, output_size))

def slice_squares_real(warped, square_size=50):
    squares = []
    for r in range(8):
        for c in range(8):
            y1 = r*square_size; x1 = c*square_size
            squares.append(warped[y1:y1+square_size, x1:x1+square_size])
    return squares

def preprocess_chessboard_real(image_path, output_size=400, display=False, **detect_kwargs):
    img = cv2.imread(image_path)
    if img is None:
        return None, None
    corners, debug = detect_board_corners_real(img, **detect_kwargs)
    if corners is None:
        return None, None
    warped = warp_board_real(img, corners, output_size)
    squares = slice_squares_real(warped, square_size=output_size//8)
    if display:
        viz = draw_corners_real(img, corners)
        tiles = [img, debug, viz, warped]
        titles = ["Original", "Debug (masked edges)", "Corner Overlay", "Warped Top-Down"]
        plt.figure(figsize=(60,20))
        for i,(im,tt) in enumerate(zip(tiles,titles),1):
            plt.subplot(1,4,i)
            if im.ndim==2: plt.imshow(im, cmap='gray')
            else: plt.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
            plt.title(tt); plt.axis('off')
        plt.tight_layout(); plt.show()
    return warped, squares

In [ ]:
warped, squares = preprocess_chessboard_real("/content/drive/MyDrive/Colab Notebooks/APS360/Project/New Photos/Overhead/IMG_6196.jpg", blur_type="bilateral", bilateral_d=7, bilateral_sigma_color=60, bilateral_sigma_space=60, min_area_frac=0.04, display=True)

In [ ]:
warped, squares = preprocess_chessboard_real("/content/drive/MyDrive/Colab Notebooks/APS360/Project/New Photos/Multiviews3/IMG_6289.jpg", blur_type="median", bilateral_d=7, bilateral_sigma_color=60, bilateral_sigma_space=60, min_area_frac=0.04, display=True)

In [ ]:
import os, json, pickle
from tqdm.notebook import tqdm
import cv2
import numpy as np

# ------------------------------
# Helper: keep only the board part of FEN (field 1) — works with digits or '0's
# ------------------------------
def board_only_fen(fen_str: str) -> str:
    # FEN can have extra fields like "w KQkq - 0 1"; we only want the 1st field
    return fen_str.split()[0]

# ------------------------------
# I/O
# ------------------------------
real_json_path = "/content/drive/MyDrive/Colab Notebooks/APS360/Project/New Photos/real_labels_2.json"
real_image_dir = "/content/drive/MyDrive/Colab Notebooks/APS360/Project/New Photos/Overhead"

with open(real_json_path, "r") as f:
    real_data = json.load(f)

save_dir = "/content/drive/MyDrive/Colab Notebooks/APS360/Project/RealBoard_Overhead_Hough"
os.makedirs(save_dir, exist_ok=True)

# ------------------------------
# Process + Save
# ------------------------------
X_real, y_real = [], []
skipped, errors = [], []

for item in tqdm(real_data, desc="Processing real images"):
    img_name = item["image"]
    gt_fen_full = item["gt_fen"]
    img_path = os.path.join(real_image_dir, img_name)

    try:
        # first pass: your new real-image pipeline (tuned defaults)
        warped, _ = preprocess_chessboard_real(
            img_path,
            display=False,
            blur_type="bilateral",
            bilateral_d=7,
            bilateral_sigma_color=60,
            bilateral_sigma_space=60,
            min_area_frac=0.04,   # you can bump to 0.06 if you get table clutter
        )

        # light second pass (optional) if the first fails
        if warped is None:
            warped, _ = preprocess_chessboard_real(
                img_path,
                display=False,
                blur_type="gaussian",
                min_area_frac=0.04,
            )

        if warped is None or np.std(warped) < 50:
            skipped.append(img_name)
            # print(f"⚠️ Skipped: {img_path} (warp failed or too low contrast)")
            continue

        # Save warped board
        out_path = os.path.join(save_dir, os.path.splitext(img_name)[0] + ".png")
        cv2.imwrite(out_path, warped)

        # Keep only the board part of FEN (digits are fine for your label fn)
        y_real.append(board_only_fen(gt_fen_full))
        X_real.append(out_path)

    except Exception as e:
        errors.append((img_name, str(e)))
        # print(f"❌ Error processing {img_path}: {e}")

# ------------------------------
# Persist lists for your loaders
# ------------------------------
pkl_path = "/content/drive/MyDrive/Colab Notebooks/APS360/Project/RealBoard_Overhead_Hough.pkl"
with open(pkl_path, "wb") as f:
    pickle.dump((X_real, y_real), f)

print(f"✅ Total saved: {len(X_real)}")
print(f"⚠️ Skipped (warp/contrast): {len(skipped)}")
print(f"❌ Errors: {len(errors)}")
if skipped:
    print("Some skipped examples:", skipped[:10])
if errors:
    print("Some errors:", errors[:5])


# EVAL ON NEW IMAGES

In [ ]:
with open("/content/drive/MyDrive/Colab Notebooks/APS360/Project/RealBoard_Overhead_Hough.pkl", "rb") as f:
    X_loaded_real, y_loaded_real = pickle.load(f)

# Only keep the board layout part (i.e., before the first space)
y_loaded_real = [fen.split()[0] for fen in y_loaded_real]

# Rest stays the same
real_transform = transforms.Compose([
    transforms.Resize((256, 256)),  # Match your model's input
    transforms.ToTensor(),
])

real_dataset = ChessboardRotDataset(X_loaded_real, y_loaded_real, transform=real_transform)

def custom_collate(batch):
    images, label_sets = zip(*batch)
    images = torch.stack(images, dim=0)
    return images, label_sets

real_loader = DataLoader(real_dataset, batch_size=8, shuffle=False, collate_fn=custom_collate)


In [ ]:
# Hardcoded FEN strings
groundTruthFen = "rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1"  # initial position
predictedFen   = "rnbqkbnr/ppp1pppp/8/3p4/4P3/5N2/PPPP1PPP/RNBQKB1R b KQkq - 1 2"  # example prediction


In [ ]:
evaluate_rotation_invariant(model, real_loader, device=device)

In [ ]:
from typing import Dict, Optional

# pretend "API" call
def _fakeApiPredictFen(imageBytes: bytes) -> str:
    # swap this with your real API request and response parsing
    # example predicted FEN (change as you like)
    return "rnbqkbnr/ppp1pppp/8/3p4/4P3/5N2/PPPP1PPP/RNBQKB1R b KQkq - 1 2"

def getFenFromImage(imagePath: str, groundTruthFen: Optional[str] = None) -> Dict[str, str]:
    """
    Reads an image, sends it to a fake API, and returns:
      - predictedFen
      - groundTruthFen (echoed from input)
      - confidence (hardcoded high value as a string percent)
    """
    with open(imagePath, "rb") as f:
        imageBytes = f.read()

    predictedFen = _fakeApiPredictFen(imageBytes)

    # hardcode a high confidence
    confidencePercent = 99.7  # adjust if you want "100"
    result = {
        "predictedFen": predictedFen,
        "groundTruthFen": groundTruthFen or "",
        "confidence": f"{confidencePercent:.1f}%",
    }
    return result

if __name__ == "__main__":
    # example usage
    gtFen = "rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1"
    out = getFenFromImage("example_board.jpg", groundTruthFen=gtFen)
    print(out)


In [ ]:

def fenBoardToArray(fen: str):
    boardPart = fen.split()[0]
    rows = boardPart.split('/')
    board = []
    for row in rows:
        expanded = []
        for ch in row:
            if ch.isdigit():
                expanded.extend(['.'] * int(ch))
            else:
                expanded.append(ch)
        board.extend(expanded)
    if len(board) != 64:
        raise ValueError("FEN board did not expand to 64 squares")
    return board

def idxToSquare(i: int) -> str:
    file = "abcdefgh"[i % 8]
    rank = str(8 - (i // 8))
    return f"{file}{rank}"

def printBoard(board):
    for r in range(8):
        row = board[r*8:(r+1)*8]
        print(8 - r, ' '.join(row))
    print("  a b c d e f g h")

def compareFens(gtFen: str, predFen: str):
    gt = fenBoardToArray(gtFen)
    pr = fenBoardToArray(predFen)

    print("Ground truth FEN:")
    print(gtFen, "\n")
    print("Predicted FEN:")
    print(predFen, "\n")

    print("Ground truth board:")
    printBoard(gt)
    print("\nPredicted board:")
    printBoard(pr)
    print()

    mismatches = []
    for i, (g, p) in enumerate(zip(gt, pr)):
        if g != p:
            mismatches.append((idxToSquare(i), g, p))

    if not mismatches:
        print("No mismatches. Boards are identical.")
    else:
        print(f"Mismatches ({len(mismatches)}):")
        for sq, g, p in mismatches:
            print(f"{sq}: gt='{g}' pred='{p}'")

if __name__ == "__main__":
    compareFens(groundTruthFen, predictedFen)
